# Ingerir Dados — Orquestrador de Ingestão (Landing → Bronze)

Ponto de entrada único que ingere as 11 tabelas de evento diário dos 4 sistemas, via `IngestorAutoloader`, da Landing Zone para a Bronze.

Diferente do orquestrador de geração (`gerar_dados.py`), não depende de `data_referencia` — o Autoloader varre toda a Landing Zone do sistema e usa o checkpoint para saber o que já foi processado, independente da data.

Cada tabela ingerida é registrada em `observability.pipeline_runs` (ADR-014, Fase C).

Referências: ADR-001 (Landing Zone), ADR-002 (streaming só aqui), ADR-012 (Autoloader), ADR-014.

In [0]:
dbutils.library.restartPython()

In [0]:
TABELAS_POR_SISTEMA = {
    "erp": ["erp_lotes_producao"],
    "crm": ["crm_pedidos", "crm_itens_pedido", "crm_atendimento"],
    "distribution": ["erp_posicoes_estoque", "erp_notas_expedicao"],
    "tms": ["tms_remessas", "tms_leituras_temperatura", "tms_comprovantes_entrega"],
    "financeiro": ["financeiro_faturas", "financeiro_contas_receber"],
}

In [0]:
dbutils.widgets.dropdown("sistema", "todos", ["todos", "crm", "erp", "distribution", "tms", "financeiro"], "Sistema")

## Leitura do Widget

Determina quais sistemas processar: todos os 4, ou um específico selecionado no dropdown — mesmo padrão de reprocessamento isolado usado no orquestrador de geração.

In [0]:
# Leitura do Widget "sistema"
sistema_selecionado = dbutils.widgets.get("sistema")
sistemas_a_processar = list(TABELAS_POR_SISTEMA.keys()) if sistema_selecionado == "todos" else [sistema_selecionado]

print(f"Sistemas a processar: {sistemas_a_processar}")

## Execução

Ingere cada tabela dos sistemas selecionados via `IngestorAutoloader`. Diferente da geração, a ordem entre sistemas não importa aqui — ingestão não tem dependência cruzada entre sistemas, só entre Landing e Bronze do mesmo sistema.

Cada tabela ingerida é registrada em `observability.pipeline_runs`.

In [0]:
# Ingestão das tabelas selecionadas, com registro de execução
from src.ingestao.ingestor_autoloader import IngestorAutoloader
from src.observabilidade.registrar_execucao import registrar_execucao_pipeline

resultados = []
for sistema in sistemas_a_processar:
    for tabela in TABELAS_POR_SISTEMA[sistema]:
        ingestor = IngestorAutoloader(spark=spark, sistema=sistema, tabela=tabela)
        resultado = ingestor.executar()
        resultados.append(resultado)
        print(resultado)

        registrar_execucao_pipeline(
            spark=spark,
            pipeline="ingerir_dados",
            item=tabela,
            status=resultado["status"],
            detalhes=resultado,
        )

print("\nResumo da ingestão:")
for r in resultados:
    print(f"  {r['tabela']}: {r['status']} — {r['arquivos_processados']} arquivo(s), {r['linhas_processadas']} linha(s)")

In [0]:
dbutils.fs.ls("/Volumes/poc_pulse_observability/landing/raw/erp/data=2026-08-12/")

In [0]:
df_bronze_check = spark.table("poc_pulse_observability.bronze.erp_lotes_producao")
df_bronze_check.filter(df_bronze_check.data == "2026-08-12").count()

In [0]:
spark.sql("DESCRIBE HISTORY poc_pulse_observability.bronze.erp_lotes_producao").show(20, truncate=False)